# 🗂️ State vs Runtime Context

---

These two mechanisms look similar on the surface — both let tools access extra data beyond the conversation messages — but they serve **fundamentally different purposes**.

## 📌 Runtime Context

> Data you **already know** *before* the conversation starts.

- 🔒 **Read-only** — the agent can read it via tools, but cannot modify it
- ⏱️ **Set at invocation time** — injected from outside via `context=`
- 💾 **Not persisted** — gone after the call ends
- 🧑‍💼 **Examples:** user profile, account settings, permissions, feature flags

## 🔄 State

> Data the agent **discovers or builds up** *during* the conversation.

- ✏️ **Mutable** — tools can write to it via `Command(update={...})`
- 🕐 **Updated mid-conversation** — evolves as the agent learns more
- 💿 **Persisted across turns** — survives multiple invocations when a `checkpointer` is used
- 🛒 **Examples:** preferences revealed mid-chat, items added to a cart, decisions made along the way

---

## ⚖️ Side-by-side comparison

| | 🔵 Runtime Context | 🟢 State |
|---|---|---|
| **Who sets it** | You (the caller) | The agent (via tools) |
| **When** | Before invocation | During the conversation |
| **Mutable?** | ❌ No | ✅ Yes |
| **Persisted across turns?** | ❌ No | ✅ Yes (with checkpointer) |
| **Use case** | Known user data upfront | Data discovered/updated mid-session |

> 💡 They're **complementary, not alternatives**. A real app would use both — runtime context for *"here's what I already know about this user"*, and state for *"here's what the agent learned during this session"*.

### ⭐️ Unlike `context`, we cannot provide `default values` for `State` ❌

### ⭐️ Similar to `context`, `state` values are accessed by `tools` at runtime ✅

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

### The `Command` function is used to `update state` 🔄

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    "Successfully updated favourite colour",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "3"}},
)

In [7]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='9f02b5e0-fbe5-4f00-8914-5f277d951920'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 141, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpUCq0rXerJuvILgVqme4gka5rPD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd730-5a29-7ce3-a1d9-1eb0171a9474-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_F8VDv7nWhbLNSDRQUuVLy1Zk', 'type': 'tool_call'}], invalid_

In [8]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='ceae359d-ca3d-424e-818e-b345851ccfc5'),
              AIMessage(content='Hello! I’m here and ready to help with whatever you need. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 142, 'total_tokens': 492, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpabEbPgrRPrB5LG87AufmX0qiku', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dd736-6a36-7550-bb78-add255f7d475-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_toke

## Read state

In [9]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "20"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='7bd2efaf-3188-41c6-9dad-f9d29a2294bd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpbKWNEOBnN5O2XirCNJcDlD4Uyc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd737-19b3-7a32-8f94-ecc2f363c64b-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_q6ntD2bPQPdTr6oKsf8gUZRI', 'type': 'tool_call'}], invalid_

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "20"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='7bd2efaf-3188-41c6-9dad-f9d29a2294bd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 347, 'prompt_tokens': 162, 'total_tokens': 509, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpbKWNEOBnN5O2XirCNJcDlD4Uyc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd737-19b3-7a32-8f94-ecc2f363c64b-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_q6ntD2bPQPdTr6oKsf8gUZRI', 'type': 'tool_call'}], invalid_